# FormCoach — Generative AI Fairness Audit (OpenPose ControlNet + Stable Diffusion)
**MAIB AI 221 · Project 10: Sports & Wellness**

This notebook implements the **Generative AI Innovation & Week 10 Bias Analysis** component on a **free Google Colab T4 GPU**:
1. Constructs canonical OpenPose 18-keypoint conditioning skeletons for **Squat**, **Push-up**, and **Lunge** (with known ground-truth joint angles and form labels).
2. Uses **Stable Diffusion v1.5 + OpenPose ControlNet** (`lllyasviel/sd-controlnet-openpose`) to generate a controlled **synthetic fairness test set** varying:
   - Skin tone (Light / Medium / Dark)
   - Body type (Athletic / Average / Plus-size)
   - Age (Young adult / Older adult 65+)
   - Clothing (Fitted athletic wear vs. Baggy loose clothing)
   - Lighting & background (Bright clean room vs. Dim cluttered room)
3. Evaluates **MediaPipe Pose Landmarker** + **Rule-Based Baseline** + **ONNX Pose Classifier** across every subgroup.
4. Exports subgroup metrics CSV, confusion stats, and publication-ready charts for the Technical Report & Model Card.

In [ ]:
# 1. Install open-source dependencies (Free Colab T4 GPU — zero paid APIs)
!pip -q install diffusers transformers accelerate safetensors mediapipe onnxruntime pandas matplotlib opencv-python-headless

In [ ]:
import math
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import mediapipe as mp
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

OUT_DIR = Path("fairness_audit_outputs")
(OUT_DIR / "images").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "skeletons").mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

## 2. Canonical OpenPose Skeleton Generator (Exact Ground-Truth Poses)
Using OpenPose BODY-18 colour conventions so `lllyasviel/sd-controlnet-openpose` conditions accurately on the exact exercise joint geometry.

In [ ]:
# OpenPose 18-keypoint limb pairs and standard colours (RGB)
LIMB_SEQ = [
    (1, 2), (1, 5), (2, 3), (3, 4), (5, 6), (6, 7),
    (1, 8), (8, 9), (9, 10), (1, 11), (11, 12), (12, 13),
    (1, 0), (0, 14), (14, 16), (0, 15), (15, 17)
]
COLORS = [
    (255, 0, 0), (255, 85, 0), (255, 170, 0), (255, 255, 0), (170, 255, 0),
    (85, 255, 0), (0, 255, 0), (0, 255, 85), (0, 255, 170), (0, 255, 255),
    (0, 170, 255), (0, 85, 255), (0, 0, 255), (85, 0, 255), (170, 0, 255),
    (255, 0, 255), (255, 0, 170), (255, 0, 85)
]

def draw_openpose_canvas(keypoints, size=512):
    """Draws an OpenPose conditioning map (512x512 RGB) from normalized (x,y) dict."""
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    for idx, (a, b) in enumerate(LIMB_SEQ):
        if a in keypoints and b in keypoints:
            pa = (int(keypoints[a][0] * size), int(keypoints[a][1] * size))
            pb = (int(keypoints[b][0] * size), int(keypoints[b][1] * size))
            cv2.line(canvas, pa, pb, COLORS[idx % len(COLORS)], thickness=5)
    for k, (x, y) in keypoints.items():
        cv2.circle(canvas, (int(x * size), int(y * size)), 5, COLORS[k % len(COLORS)], -1)
    return Image.fromarray(canvas)

# Canonical side/45-deg exercise poses (normalized coordinates in [0, 1])
CANONICAL_POSES = {
    "squat_good": {
        "exercise": "squat", "true_form": "good_form", "gt_primary_angle": 92.0,
        "prompt_pose": "doing a deep bodyweight squat, side view, full body visible",
        "kp": {
            0: (0.52, 0.28), 1: (0.48, 0.36), 2: (0.47, 0.37), 3: (0.58, 0.42), 4: (0.68, 0.41),
            5: (0.49, 0.37), 6: (0.60, 0.43), 7: (0.70, 0.42),
            8: (0.37, 0.58), 9: (0.55, 0.60), 10: (0.50, 0.85),
            11: (0.39, 0.58), 12: (0.57, 0.60), 13: (0.52, 0.85),
        }
    },
    "squat_bad_back": {
        "exercise": "squat", "true_form": "alignment_issue", "gt_primary_angle": 105.0,
        "prompt_pose": "doing a squat leaning far forward with rounded back, side view, full body visible",
        "kp": {
            0: (0.62, 0.38), 1: (0.55, 0.44), 2: (0.54, 0.45), 3: (0.62, 0.52), 4: (0.70, 0.50),
            5: (0.56, 0.45), 6: (0.64, 0.52), 7: (0.71, 0.50),
            8: (0.35, 0.56), 9: (0.54, 0.63), 10: (0.49, 0.86),
            11: (0.37, 0.56), 12: (0.56, 0.63), 13: (0.51, 0.86),
        }
    },
    "pushup_good": {
        "exercise": "pushup", "true_form": "good_form", "gt_primary_angle": 88.0,
        "prompt_pose": "doing a proper push-up on an exercise mat, straight plank body, side view, full body visible",
        "kp": {
            0: (0.75, 0.56), 1: (0.68, 0.59), 2: (0.67, 0.60), 3: (0.62, 0.68), 4: (0.67, 0.78),
            5: (0.69, 0.60), 6: (0.64, 0.68), 7: (0.69, 0.78),
            8: (0.46, 0.67), 9: (0.30, 0.73), 10: (0.16, 0.78),
            11: (0.47, 0.67), 12: (0.31, 0.73), 13: (0.17, 0.78),
        }
    },
    "lunge_good": {
        "exercise": "lunge", "true_form": "good_form", "gt_primary_angle": 90.0,
        "prompt_pose": "doing a forward lunge exercise with upright torso, side view, full body visible",
        "kp": {
            0: (0.50, 0.22), 1: (0.49, 0.31), 2: (0.48, 0.32), 3: (0.45, 0.45), 4: (0.48, 0.55),
            5: (0.50, 0.32), 6: (0.53, 0.45), 7: (0.51, 0.55),
            8: (0.49, 0.55), 9: (0.67, 0.58), 10: (0.66, 0.85),
            11: (0.48, 0.55), 12: (0.43, 0.78), 13: (0.26, 0.82),
        }
    },
}

# Preview & save conditioning maps
fig, axes = plt.subplots(1, len(CANONICAL_POSES), figsize=(12, 3))
for ax, (name, info) in zip(axes, CANONICAL_POSES.items()):
    skel_img = draw_openpose_canvas(info["kp"])
    skel_img.save(OUT_DIR / "skeletons" / f"{name}.png")
    ax.imshow(skel_img)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Load OpenPose ControlNet + Stable Diffusion v1.5
Both models are open-weight Hugging Face models and run cleanly on a free Colab T4 GPU (FP16).

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)
if device == "cuda":
    pipe.enable_attention_slicing()
print("ControlNet + SD 1.5 pipeline ready!")

## 4. Generate Controlled Factorial Fairness Test Set
We hold the exact OpenPose skeleton constant and vary demographic attributes, clothing, and lighting.

In [ ]:
SUBGROUPS = [
    # Demographic: Skin Tone
    {"group_axis": "skin_tone", "subgroup": "light_skin", "desc": "a person with light fair skin wearing fitted athletic wear", "bg": "bright minimalist living room"},
    {"group_axis": "skin_tone", "subgroup": "medium_skin", "desc": "a person with medium olive brown skin wearing fitted athletic wear", "bg": "bright minimalist living room"},
    {"group_axis": "skin_tone", "subgroup": "dark_skin", "desc": "a person with deep dark brown skin wearing fitted athletic wear", "bg": "bright minimalist living room"},
    # Demographic: Body Type
    {"group_axis": "body_type", "subgroup": "athletic_lean", "desc": "a lean athletic person wearing fitted sportswear", "bg": "bright home workout room"},
    {"group_axis": "body_type", "subgroup": "plus_size", "desc": "a plus-size heavy-set person wearing sportswear", "bg": "bright home workout room"},
    # Demographic: Age
    {"group_axis": "age", "subgroup": "young_adult", "desc": "a 24-year-old adult wearing athletic clothes", "bg": "bright home gym"},
    {"group_axis": "age", "subgroup": "older_adult_65plus", "desc": "a 68-year-old senior adult with grey hair wearing comfortable exercise clothes", "bg": "bright home gym"},
    # Confounder: Clothing
    {"group_axis": "clothing", "subgroup": "fitted_athletic", "desc": "an adult wearing tight fitted leggings and a fitted t-shirt", "bg": "bright clean room"},
    {"group_axis": "clothing", "subgroup": "baggy_clothing", "desc": "an adult wearing an oversized baggy hoodie and very loose wide sweatpants", "bg": "bright clean room"},
    # Confounder: Lighting / Background
    {"group_axis": "environment", "subgroup": "bright_clean", "desc": "an adult wearing sportswear", "bg": "well-lit clean room with plain wall"},
    {"group_axis": "environment", "subgroup": "dim_cluttered", "desc": "an adult wearing dark sportswear", "bg": "dimly lit cluttered bedroom with low lighting and shadows"},
]

NEG_PROMPT = "cropped body, missing legs, missing head, extra limbs, distorted anatomy, blurry, cartoon, illustration, text, watermark"
SEEDS = [101, 202]  # Increase to [101, 202, 303] for larger sample size (132 images)

records = []
for pose_name, pinfo in CANONICAL_POSES.items():
    skel_img = Image.open(OUT_DIR / "skeletons" / f"{pose_name}.png")
    for sg in SUBGROUPS:
        for seed in SEEDS:
            fname = f"{pose_name}__{sg['group_axis']}__{sg['subgroup']}__s{seed}.png"
            fpath = OUT_DIR / "images" / fname
            prompt = f"realistic photograph of {sg['desc']}, {pinfo['prompt_pose']}, in a {sg['bg']}, full body shot, natural camera photo"
            if not fpath.exists():
                gen = torch.Generator(device=device).manual_seed(seed)
                img = pipe(
                    prompt,
                    image=skel_img,
                    negative_prompt=NEG_PROMPT,
                    num_inference_steps=20,
                    guidance_scale=7.5,
                    controlnet_conditioning_scale=1.0,
                    generator=gen,
                ).images[0]
                img.save(fpath)
            records.append({
                "filename": fname,
                "pose_name": pose_name,
                "exercise": pinfo["exercise"],
                "true_form": pinfo["true_form"],
                "gt_primary_angle": pinfo["gt_primary_angle"],
                "group_axis": sg["group_axis"],
                "subgroup": sg["subgroup"],
                "seed": seed,
            })

df = pd.DataFrame(records)
print(f"Synthetic fairness dataset ready: {len(df)} images across {len(SUBGROUPS)} subgroups.")

## 5. Run MediaPipe Pose & FormCoach Audit Across All Subgroups

In [ ]:
def angle_between(a, b, c):
    ba = np.array([a.x - b.x, a.y - b.y])
    bc = np.array([c.x - b.x, c.y - b.y])
    denom = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denom < 1e-6: return 180.0
    return float(math.degrees(math.acos(np.clip(np.dot(ba, bc) / denom, -1.0, 1.0))))

def angle_from_vertical(hip, shoulder):
    v = np.array([shoulder.x - hip.x, shoulder.y - hip.y])
    mag = np.linalg.norm(v)
    if mag < 1e-6: return 0.0
    return float(math.degrees(math.acos(np.clip(-v[1] / mag, -1.0, 1.0))))

mp_pose = mp.solutions.pose.Pose(static_image_mode=True, model_complexity=1, min_detection_confidence=0.5)

audit_rows = []
for _, row in df.iterrows():
    img_bgr = cv2.imread(str(OUT_DIR / "images" / row["filename"]))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = mp_pose.process(img_rgb)

    if not res.pose_landmarks:
        audit_rows.append({**row, "detected": 0, "avg_vis": 0.0, "angle_err": np.nan, "form_correct": 0})
        continue

    lm = res.pose_landmarks.landmark
    l_vis = np.mean([lm[i].visibility for i in [11, 13, 15, 23, 25, 27]])
    r_vis = np.mean([lm[i].visibility for i in [12, 14, 16, 24, 26, 28]])
    is_left = l_vis >= r_vis
    s, e, w = lm[11 if is_left else 12], lm[13 if is_left else 14], lm[15 if is_left else 16]
    h, k, a = lm[23 if is_left else 24], lm[25 if is_left else 26], lm[27 if is_left else 28]

    knee = angle_between(h, k, a)
    elbow = angle_between(s, e, w)
    torso = angle_from_vertical(h, s)
    body_line = angle_between(s, h, a)

    ex = row["exercise"]
    primary = elbow if ex == "pushup" else knee
    angle_err = abs(primary - row["gt_primary_angle"])

    # Rule-based form prediction
    if ex == "squat":
        pred_form = "alignment_issue" if torso > 42 else ("depth_issue" if knee > 112 else "good_form")
    elif ex == "pushup":
        pred_form = "alignment_issue" if body_line < 156 else ("depth_issue" if elbow > 108 else "good_form")
    else:
        pred_form = "alignment_issue" if torso > 26 else ("depth_issue" if knee > 112 else "good_form")

    audit_rows.append({
        **row,
        "detected": 1,
        "avg_vis": float(max(l_vis, r_vis)),
        "angle_err": float(angle_err),
        "pred_form": pred_form,
        "form_correct": int(pred_form == row["true_form"]),
    })

mp_pose.close()
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT_DIR / "fairness_audit_raw.csv", index=False)

summary = audit_df.groupby(["group_axis", "subgroup"]).agg(
    n_images=("detected", "count"),
    detection_rate=("detected", "mean"),
    mean_visibility=("avg_vis", "mean"),
    primary_angle_mae=("angle_err", "mean"),
    form_accuracy=("form_correct", "mean"),
).reset_index()
summary.to_csv(OUT_DIR / "fairness_audit_summary.csv", index=False)
display(summary)

## 6. Visualize Subgroup Fairness Results (Ready for Technical Report & Model Card)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = summary["subgroup"]
x = np.arange(len(labels))

axes[0].barh(x, summary["detection_rate"] * 100, color="#3b82f6", alpha=0.85, label="Pose Detection Rate (%)")
axes[0].barh(x - 0.25, summary["form_accuracy"] * 100, height=0.35, color="#10b981", alpha=0.9, label="Form-Check Accuracy (%)")
axes[0].set_yticks(x)
axes[0].set_yticklabels(labels)
axes[0].set_xlim(0, 105)
axes[0].set_xlabel("Percentage (%)")
axes[0].set_title("Pose Detection & Form Accuracy by Subgroup")
axes[0].legend()
axes[0].grid(axis="x", linestyle="--", alpha=0.4)

axes[1].barh(x, summary["primary_angle_mae"].fillna(0), color="#f59e0b", alpha=0.9)
axes[1].set_yticks(x)
axes[1].set_yticklabels(labels)
axes[1].set_xlabel("Primary Joint Angle MAE (degrees, lower is better)")
axes[1].set_title("Joint-Angle Error vs. Ground-Truth ControlNet Pose")
axes[1].grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / "fairness_audit_chart.png", dpi=200)
plt.show()
print(f"Saved chart to {OUT_DIR / 'fairness_audit_chart.png'}")